# Prompt Templates and Variables Tutorial (Using Jinja2)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/prompt-templates-variables-jinja2.ipynb)

## Overview

This tutorial provides a comprehensive introduction to creating and using prompt templates with variables in the context of AI language models. It focuses on leveraging Python and the Jinja2 templating engine to create flexible, reusable prompt structures that can incorporate dynamic content. The tutorial demonstrates how to interact with open-source language models via HuggingFace Transformers, running locally in Google Colab.

## Motivation

As AI language models become increasingly sophisticated, the ability to craft effective prompts becomes crucial for obtaining desired outputs. Prompt templates and variables offer several advantages:

1. **Reusability**: Templates can be reused across different contexts, saving time and ensuring consistency.
2. **Flexibility**: Variables allow for dynamic content insertion, making prompts adaptable to various scenarios.
3. **Complexity Management**: Templates can handle complex structures, including conditional logic and loops, enabling more sophisticated interactions with AI models.
4. **Scalability**: As applications grow, well-structured templates make it easier to manage and maintain large numbers of prompts.

This tutorial aims to equip learners with the knowledge and skills to create powerful, flexible prompt templates, enhancing their ability to work effectively with AI language models.

## Key Components

The tutorial covers several key components:

1. **PromptTemplate Class**: A custom class that wraps Jinja2's Template class, providing a simple interface for creating and using templates.
2. **Jinja2 Templating**: Utilization of Jinja2 for advanced templating features, including variable insertion, conditional statements, and loops.
3. **HuggingFace Transformers Integration**: Using open-source models locally for generating responses from prompt templates.
4. **Variable Handling**: Techniques for incorporating variables into templates and managing dynamic content.
5. **Conditional Logic**: Implementation of if-else statements within templates to create context-aware prompts.
6. **Advanced Formatting**: Methods for structuring complex prompts, including list formatting and multi-part instructions.

## Method Details

The tutorial employs a step-by-step approach to introduce and demonstrate prompt templating concepts:

1. **Setup and Environment**: The lesson begins by setting up the necessary libraries, including Jinja2 and HuggingFace Transformers, and loading a quantized open-source model in Google Colab.

2. **Basic Template Creation**: Introduction to creating simple templates with single and multiple variables using the custom PromptTemplate class.

3. **Variable Insertion**: Demonstration of how to insert variables into templates using Jinja2's `{{ variable }}` syntax.

4. **Conditional Content**: Exploration of using if-else statements in templates to create prompts that adapt based on provided variables.

5. **List Processing**: Techniques for handling lists of items within templates, including iteration and formatting.

6. **Advanced Templating**: Demonstration of more complex template structures, including nested conditions, loops, and multi-part prompts.

7. **Dynamic Instruction Generation**: Creation of templates that can generate structured instructions based on multiple input variables.

8. **Model Integration**: Throughout the tutorial, examples show how to use the templates with a locally-running HuggingFace model to generate responses.

The methods are presented with practical examples, progressing from simple to more complex use cases. Each concept is explained theoretically and then demonstrated with a practical application.

## Conclusion

This tutorial provides a solid foundation in creating and using prompt templates with variables, leveraging the power of Jinja2 for advanced templating features. By the end of the lesson, learners will have gained:

1. Understanding of the importance and applications of prompt templates in AI interactions.
2. Practical skills in creating reusable, flexible prompt templates.
3. Knowledge of how to incorporate variables and conditional logic into prompts.
4. Experience in structuring complex prompts for various use cases.
5. Insight into integrating templated prompts with open-source models via HuggingFace Transformers.

These skills enable more sophisticated and efficient interactions with AI language models, opening up possibilities for creating more advanced, context-aware AI applications. The techniques learned can be applied to a wide range of scenarios, from simple query systems to complex, multi-turn conversational agents.

## Setup

Install dependencies, load a quantized open-source model (Qwen3-8B) via HuggingFace Transformers, and define a `generate()` helper for local inference in Google Colab.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch jinja2

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from jinja2 import Template

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def get_completion(prompt, temperature=0.1):
    """Convenience wrapper: send a single user prompt and return the model response."""
    messages = [{"role": "user", "content": prompt}]
    return generate(messages, temperature=temperature, do_sample=temperature > 0)

## Why Templates Beat String Concatenation

Before diving into code, it's worth understanding **why** prompt templates exist at all. Python f-strings and `.format()` can build any string — so why add a template engine?

### The Core Problem

Prompt engineering isn't just about writing text — it's about producing **structured input** that a model tokenizes into a predictable sequence. Consider what happens at the system level:

1. **Your prompt** → gets **tokenized** into integer IDs → fed to the model's embedding layer → processed through transformer layers → produces output tokens.
2. The model has learned statistical patterns over **token sequences**, not raw text. Formatting consistency matters because the same semantic content can tokenize differently depending on whitespace, punctuation, and structure.

### Why Templates Win

| Concern | f-strings / .format() | Jinja2 Templates |
|---|---|---|
| **Variable binding** | Inline in Python code | Declarative, separated from logic |
| **Reusability** | Copy-paste the f-string | Render the same template object with different data |
| **Safety** | User input can break formatting or inject code | Variables are escaped; sandboxing available |
| **Complex logic** | Python code mixed with string building | Conditionals and loops inside the template |
| **Separation of concerns** | Prompt IS the code | Prompt is data; code just fills variables |

The key insight: **templates separate the prompt structure (what the model sees) from the data (what changes per request)**. This mirrors how web frameworks separate HTML templates from application logic — and for the same reasons.


In [ ]:
# === f-string vs Jinja2: What's actually different? ===
from jinja2 import Template

topic = "machine learning"
audience = "beginner"

# Method 1: f-string
fstring_prompt = f"Explain {topic} to a {audience} audience. Be concise and use examples."

# Method 2: Jinja2 template (reusable object)
template = Template("Explain {{ topic }} to a {{ audience }} audience. Be concise and use examples.")
jinja_prompt = template.render(topic=topic, audience=audience)

print("f-string result:", repr(fstring_prompt))
print("Jinja2 result:  ", repr(jinja_prompt))
print("Identical?", fstring_prompt == jinja_prompt)

# Tokenize both to prove they produce the same token sequence
fstring_tokens = tokenizer.encode(fstring_prompt)
jinja_tokens = tokenizer.encode(jinja_prompt)

print(f"\nf-string tokens ({len(fstring_tokens)}): {fstring_tokens[:15]}...")
print(f"Jinja2 tokens   ({len(jinja_tokens)}): {jinja_tokens[:15]}...")
print("Token sequences identical?", fstring_tokens == jinja_tokens)

# With simple inputs, both are identical. The advantage is structural:
# the Jinja2 template object can be reused, shared, stored, and rendered safely.
reuse_prompt = template.render(topic="quantum computing", audience="expert")
print(f"\nReused template: {reuse_prompt}")


In [ ]:
# === The Bug That Templates Prevent ===
# User input with curly braces breaks str.format() but NOT Jinja2

user_input = "Explain Python's {dictionary} syntax and {{nested}} braces"

# str.format() treats { } as placeholders — this BREAKS
format_string = "Answer this question: {user_question}\nProvide a detailed response."
try:
    result = format_string.format(user_question=user_input)
    print("str.format result:", result)
except KeyError as e:
    print(f"str.format() BROKE with KeyError: {e}")
    print("  -> Curly braces in user input were interpreted as format placeholders!")

# Jinja2 only recognizes {{ }} as placeholders — user { } are literal text
jinja_template = Template("Answer this question: {{ user_question }}\nProvide a detailed response.")
result = jinja_template.render(user_question=user_input)
print(f"\nJinja2 result: {result}")
print("\n✓ Jinja2 uses {{ }} syntax, so single { } in user content are safe.")
print("  This is critical when building prompts from untrusted user input.")


## 1. Creating Reusable Prompt Templates

We'll create a PromptTemplate class that uses Jinja2 for templating:

In [ ]:
class PromptTemplate:
    ''' A class to represent a template for generating prompts with variables
    Attributes:
        template (str): The template string with variables
        input_variables (list): A list of the variable names in the template
    '''
    def __init__(self, template, input_variables):
        self.template = Template(template)
        self.input_variables = input_variables
    
    def format(self, **kwargs):
        return self.template.render(**kwargs)

# Simple template with one variable
simple_template = PromptTemplate(
    template="Provide a brief explanation of {{ topic }}.",
    input_variables=["topic"]
)

# More complex template with multiple variables
complex_template = PromptTemplate(
    template="Explain the concept of {{ concept }} in the field of {{ field }} to a {{ audience }} audience, concisely.",
    input_variables=["concept", "field", "audience"]
)

# Using the simple template
print("Simple Template Result:")
prompt = simple_template.format(topic="photosynthesis")
print(get_completion(prompt))

print("\n" + "-"*50 + "\n")

# Using the complex template
print("Complex Template Result:")
prompt = complex_template.format(
    concept="neural networks",
    field="artificial intelligence",
    audience="beginner"
)
print(get_completion(prompt))

## Jinja2 Under the Hood

Here's a fact that connects everything: **HuggingFace Transformers uses Jinja2 as its chat template engine**. When you call `tokenizer.apply_chat_template()`, it's rendering a Jinja2 template stored in `tokenizer.chat_template`.

This means the same Jinja2 you use to build prompts is the **exact same engine** that formats messages into the model's expected `<|im_start|>user\n...<|im_end|>` structure. Understanding Jinja2 isn't just a convenience — it's understanding the machinery that sits between your messages and the model's token input.


In [ ]:
# === The chat template IS a Jinja2 template ===
# Every HuggingFace tokenizer stores a Jinja2 template that controls
# how chat messages are formatted before tokenization.

print("=== tokenizer.chat_template (raw Jinja2) ===\n")
print(tokenizer.chat_template)

print("\n\n=== What it produces ===\n")
sample_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]
formatted = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
print(formatted)
print("\nThis formatted string is what actually gets tokenized and fed to the model.")


In [ ]:
# === Jinja2's Abstract Syntax Tree (AST) ===
# Templates aren't interpreted character-by-character — they're COMPILED into a tree

from jinja2 import Environment

env = Environment()

# Parse a simple template into its AST
simple_source = "Hello {{ name }}, you are a {{ role }}."
ast = env.parse(simple_source)

print(f"Template: {simple_source!r}\n")
print("AST body (the compiled structure):")
for i, node in enumerate(ast.body):
    print(f"  Node {i}: {type(node).__name__}")
    if hasattr(node, 'nodes'):
        for j, child in enumerate(node.nodes):
            if hasattr(child, 'data'):
                print(f"    Child {j}: {type(child).__name__} -> {child.data!r}")
            elif hasattr(child, 'name'):
                print(f"    Child {j}: {type(child).__name__} -> variable '{child.name}'")

# Now parse the actual chat template — see how complex real templates are
print("\n=== Chat template AST (top-level nodes) ===")
chat_ast = env.parse(tokenizer.chat_template)
for i, node in enumerate(chat_ast.body[:8]):
    node_type = type(node).__name__
    extra = ""
    if hasattr(node, 'test'):
        extra = f" (condition: {type(node.test).__name__})"
    print(f"  Node {i}: {node_type}{extra}")
if len(chat_ast.body) > 8:
    print(f"  ... ({len(chat_ast.body)} total top-level nodes)")

print("\nThe chat template compiles into For loops, If branches, and Output nodes —")
print("the same constructs you use in your own Jinja2 templates.")


### The Connection

When you write a Jinja2 template for your prompts, you're using the **same language and engine** that HuggingFace uses internally to format chat messages. This means:

- **Your templates** control what the user/assistant content says
- **The chat template** controls how that content is wrapped with special tokens (`<|im_start|>`, `<|im_end|>`, etc.)
- **Both are Jinja2** — both compile to ASTs, both support conditionals, loops, and filters
- Understanding one helps you understand (and customize) the other

This is why learning Jinja2 properly matters: you're not just learning a string formatting tool — you're learning the language that controls how messages become tokens.


## 2. Using Variables for Dynamic Content

Now let's explore more advanced uses of variables, including conditional content:

In [ ]:
# Template with conditional content
conditional_template = PromptTemplate(
    template="My name is {{ name }} and I am {{ age }} years old. "
              "{% if profession %}I work as a {{ profession }}.{% else %}I am currently not employed.{% endif %} "
              "Can you give me career advice based on this information? answer concisely.",
    input_variables=["name", "age", "profession"]
)

# Using the conditional template
print("Conditional Template Result (with profession):")
prompt = conditional_template.format(
    name="Alex",
    age="28",
    profession="software developer"
)
print(get_completion(prompt))

print("\nConditional Template Result (without profession):")
prompt = conditional_template.format(
    name="Sam",
    age="22",
    profession=""
)
print(get_completion(prompt))

print("\n" + "-"*50 + "\n")


In [ ]:
# Template for list processing
list_template = PromptTemplate(
    template="Categorize these items into groups: {{ items }}. Provide the categories and the items in each category.",
    input_variables=["items"]
)

# Using the list template
print("List Template Result:")
prompt = list_template.format(
    items="apple, banana, carrot, hammer, screwdriver, pliers, novel, textbook, magazine"
)
print(get_completion(prompt))

## Template Security

Template injection is a real security concern. When templates process untrusted user input, an attacker might craft input that executes arbitrary code. This matters for prompt engineering because prompts often include user-provided text.

### The Threat Model

- **String formatting attacks**: Python's `str.format()` and `format_map()` can leak object attributes if an attacker controls the format string
- **eval/exec injection**: Never use `eval()` to process user-provided template-like strings
- **Template injection**: Even Jinja2 in its default mode allows some attribute access that could be exploited

The solution: Jinja2 provides a `SandboxedEnvironment` that restricts what templates can access, making it safe to render templates containing untrusted content.


In [ ]:
# === Security: How format strings can leak data ===

# DANGER 1: str.format() can traverse object attributes
class DatabaseConfig:
    host = "prod-db.internal"
    password = "super_secret_123"

config = DatabaseConfig()

# An attacker who controls the format string can access attributes:
attacker_format_string = "Server info: {cfg.host}, password: {cfg.password}"
leaked = attacker_format_string.format(cfg=config)
print(f"Leaked via str.format(): {leaked}")
print("  -> Attacker accessed .host and .password through format string!\n")

# DANGER 2: eval() with user input — NEVER do this
malicious = "__import__('os').getcwd()"
print("eval() with untrusted input:")
try:
    result = eval(malicious)
    print(f"  Executed arbitrary code! Got: {result}")
    print("  An attacker could read files, delete data, or worse.\n")
except Exception as e:
    print(f"  Error: {e}\n")

# SAFE: Jinja2 only substitutes declared variables
from jinja2 import Template
safe = Template("Server info: {{ cfg.host }}")
result = safe.render(cfg={"host": "prod-db.internal"})
print(f"Jinja2 (safe dict access): {result}")
print("  Jinja2 treats cfg as a dict/object lookup — no arbitrary code execution.")


In [ ]:
# === Jinja2 SandboxedEnvironment ===
from jinja2.sandbox import SandboxedEnvironment

sandbox = SandboxedEnvironment()

# Normal rendering works perfectly
safe_template = sandbox.from_string("Hello {{ name }}, your role is {{ role }}.")
print("Safe rendering:", safe_template.render(name="Alice", role="engineer"))

# But attempts to access dangerous Python internals are BLOCKED
print("\nAttempting to access Python internals via template...")
try:
    dangerous = sandbox.from_string("{{ ''.__class__.__mro__[1].__subclasses__() }}")
    result = dangerous.render()
    print(f"  Result: {result}")
except Exception as e:
    print(f"  ✗ BLOCKED: {type(e).__name__}: {e}")

# Accessing private/dunder attributes is restricted
print("\nAttempting private attribute access...")
try:
    private_tmpl = sandbox.from_string("{{ items.__class__().__init__.__globals__ }}")
    result = private_tmpl.render(items=[1, 2, 3])
    print(f"  Result: {result}")
except Exception as e:
    print(f"  ✗ BLOCKED: {type(e).__name__}: {e}")

print("\n=== Best Practice ===")
print("Use SandboxedEnvironment when rendering templates that include user-provided content.")
print("Use regular Template/Environment when you fully control the template source.")


## Advanced Template Techniques

Let's explore some more advanced techniques for working with prompt templates and variables:

In [ ]:
# Template with formatted list
list_format_template = PromptTemplate(
    template="Analyze the following list of items:\n"
              "{% for item in items.split(',') %}"
              "- {{ item.strip() }}\n"
              "{% endfor %}"
              "\nProvide a summary of the list and suggest any patterns or groupings.",
    input_variables=["items"]
)


# Using the formatted list template
print("Formatted List Template Result:")
prompt = list_format_template.format(
    items="Python, JavaScript, HTML, CSS, React, Django, Flask, Node.js"
)
print(get_completion(prompt))

print("\n" + "-"*50 + "\n")

In [ ]:
# Template with dynamic instructions
dynamic_instruction_template = PromptTemplate(
    template="Task: {{ task }}\n"
              "Context: {{ context }}\n"
              "Constraints: {{ constraints }}\n\n"
              "Please provide a solution that addresses the task, considers the context, and adheres to the constraints.",
    input_variables=["task", "context", "constraints"]
)

# Using the dynamic instruction template
print("Dynamic Instruction Template Result:")
prompt = dynamic_instruction_template.format(
    task="Design a logo for a tech startup",
    context="The startup focuses on AI-driven healthcare solutions",
    constraints="Must use blue and green colors, and should be simple enough to be recognizable when small"
)
print(get_completion(prompt))

## Expanded Template Patterns

The following patterns show how Jinja2's features map to common prompt engineering needs. Each pattern addresses a specific structural challenge that simple string formatting cannot handle cleanly.


In [ ]:
# === Pattern: Composing Prompts from Reusable Fragments ===
# Real-world prompts share structure. Build them from composable parts.

from jinja2 import Environment

env = Environment()

# Define reusable fragments
role_fragment = "You are a {{ role }} specializing in {{ domain }}."
format_fragment = "Format your response as: {{ format_style }}."
constraint_fragment = "{% if max_words %}Keep your response under {{ max_words }} words.{% endif %}"

# Compose into a master template by combining fragments
master_source = (
    role_fragment + "\n\n"
    "User Request: {{ request }}\n\n"
    + constraint_fragment + "\n"
    + format_fragment
)

master = env.from_string(master_source)

prompt = master.render(
    role="senior data scientist",
    domain="NLP",
    request="Explain attention mechanisms in transformers",
    format_style="a numbered list with brief explanations",
    max_words=200
)
print("=== Composed Prompt ===")
print(prompt)
print("\n--- Model Response ---")
print(get_completion(prompt))


In [ ]:
# === Pattern: Adaptive Prompts with {% if %} Blocks ===
# The prompt structure itself changes based on context

from jinja2 import Template

adaptive = Template("""
{%- if expertise == "beginner" -%}
Explain in simple terms with everyday analogies. Avoid jargon.
{%- elif expertise == "intermediate" -%}
Use technical terms but briefly define any advanced concepts.
{%- elif expertise == "expert" -%}
Be precise and use domain-specific terminology freely.
{%- endif %}

Topic: {{ topic }}
{%- if include_examples %}

Provide {{ num_examples | default(2) }} concrete examples.
{%- endif %}
{%- if context %}

Additional context: {{ context }}
{%- endif %}
""".strip())

# Same template, different audiences — structure adapts automatically
for level in ["beginner", "expert"]:
    prompt = adaptive.render(
        expertise=level,
        topic="gradient descent optimization",
        include_examples=True,
        num_examples=2,
        context="" if level == "beginner" else "Focus on Adam vs SGD with momentum"
    )
    print(f"\n{'='*50}")
    print(f"  {level.upper()} PROMPT")
    print(f"{'='*50}")
    print(prompt)
    print(f"\n--- Response ---")
    print(get_completion(prompt))


In [ ]:
# === Pattern: Dynamic Few-Shot with {% for %} Loops ===
# Build few-shot examples from DATA, not hardcoded strings

from jinja2 import Template

few_shot = Template("""Classify the sentiment of the given text as positive, negative, or neutral.

{% for ex in examples -%}
Text: "{{ ex.text }}"
Sentiment: {{ ex.label }}

{% endfor -%}
Text: "{{ query }}"
Sentiment:""")

# Examples are data — easy to modify, A/B test, or load from a file
examples = [
    {"text": "This product exceeded all my expectations!", "label": "positive"},
    {"text": "Shipping was late and the item arrived damaged.", "label": "negative"},
    {"text": "The package arrived on Tuesday.", "label": "neutral"},
]

queries = [
    "I absolutely love how intuitive this new interface is!",
    "The meeting has been rescheduled to 3 PM.",
]

for q in queries:
    prompt = few_shot.render(examples=examples, query=q)
    response = get_completion(prompt, temperature=0.1)
    print(f"Query: {q!r}")
    print(f"Model: {response.strip()}")
    print()

# The power of loops: change example count without touching the template
print("=== With 1 example instead of 3 ===")
prompt_1shot = few_shot.render(examples=examples[:1], query="Terrible experience, never again.")
print(prompt_1shot)
print(f"\nModel: {get_completion(prompt_1shot, temperature=0.1).strip()}")


In [ ]:
# === Pattern: Template Inheritance ===
# Like class inheritance: define a base structure, override specific blocks

from jinja2 import Environment, DictLoader

loader = DictLoader({
    # Base template defines the overall structure with overridable blocks
    'base': """
{%- block system -%}You are a helpful AI assistant.{%- endblock %}

{% block context %}{% endblock %}
{% block task %}{% endblock %}

{%- block format -%}
Respond clearly and concisely.
{%- endblock -%}""",

    # Analysis template inherits base, overrides specific blocks
    'analysis': """
{%- extends "base" -%}
{%- block system -%}You are an expert analyst with deep domain knowledge.{%- endblock -%}
{%- block context -%}
Domain: {{ domain }}
Data: {{ data }}
{%- endblock -%}
{%- block task -%}
Identify the top 3 key insights from this data.
{%- endblock -%}
{%- block format -%}
Format: [Insight N]: Title — Explanation (1-2 sentences)
{%- endblock -%}""",

    # Comparison template: same base, different overrides
    'comparison': """
{%- extends "base" -%}
{%- block system -%}You are a balanced evaluator considering multiple perspectives.{%- endblock -%}
{%- block task -%}
Compare {{ item_a }} vs {{ item_b }} for: {{ use_case }}.
List strengths of each, key differences, and a recommendation.
{%- endblock -%}""",
})

env = Environment(loader=loader)

# Render analysis prompt
prompt = env.get_template('analysis').render(
    domain="E-commerce",
    data="Q4 sales up 23%, mobile traffic +45%, cart abandonment at 68%, avg order value $47"
)
print("=== Analysis Prompt (inherits base) ===")
print(prompt)
print("\n--- Response ---")
print(get_completion(prompt))

print("\n" + "=" * 60 + "\n")

# Render comparison prompt
prompt = env.get_template('comparison').render(
    item_a="PostgreSQL", item_b="MongoDB",
    use_case="real-time analytics dashboard"
)
print("=== Comparison Prompt (inherits base) ===")
print(prompt)
print("\n--- Response ---")
print(get_completion(prompt))


In [ ]:
# === When Templates HURT: Over-Engineering ===
from jinja2 import Template

# This template crams too much logic into Jinja2 — hard to read and debug
over_engineered = Template("""
{%- set greeting = "Hello" if time_of_day == "morning" else "Good evening" if time_of_day == "evening" else "Hi" -%}
{%- set prefix = "Please " if formal else "" -%}
{%- set closing = "Thank you for your assistance." if formal else "Thanks!" -%}
{{ greeting }}! {{ prefix }}help me with: {{ task }}. {{ closing }}
""".strip())

prompt_j = over_engineered.render(time_of_day="morning", formal=True, task="writing an email")
print("Over-engineered Jinja2 template:")
print(prompt_j)

# The same thing as plain Python — MUCH clearer
time_of_day, formal, task = "morning", True, "writing an email"
greeting = "Hello" if time_of_day == "morning" else "Good evening" if time_of_day == "evening" else "Hi"
prefix = "Please " if formal else ""
closing = "Thank you for your assistance." if formal else "Thanks!"
prompt_f = f"{greeting}! {prefix}help me with: {task}. {closing}"

print("\nEquivalent f-string:")
print(prompt_f)
print(f"\nIdentical output? {prompt_j == prompt_f}")
print("\nThe f-string version is easier to read, debug, and maintain.")
print("Don't use templates for one-off prompts where Python is clearer.")


## When to Use Templates vs Plain Python

### ✅ Use Jinja2 Templates When:
- **Reusability** — The same prompt structure is rendered with different data across your application
- **User-facing content** — User input flows into prompts (use `SandboxedEnvironment` for safety)
- **Complex structure** — Few-shot examples, conditional sections, or iterative prompt building
- **Separation of concerns** — Prompt authors (non-engineers) edit templates independently of code
- **Chat template customization** — You need to understand or modify HuggingFace's chat formatting

### ❌ Use f-strings / Plain Python When:
- **One-off prompts** — Simple, single-use prompts that won't be reused
- **Heavy logic** — Complex conditional logic that's clearer in Python than in template syntax
- **Debugging** — During development, f-strings are easier to step through in a debugger
- **Performance-critical paths** — f-strings are marginally faster (though this rarely matters for LLM calls)

### The Rule of Thumb
> If you find yourself **copy-pasting a prompt and changing a few words**, that's a template waiting to happen.
> If you find yourself **writing complex Jinja2 logic** that would be a one-liner in Python, just use Python.
